# Knowledge Graph Extraction — Pipeline 4 Passes

**Strategy:**
- **Pass 1** : Scan chunk → detect which entity types are present (labels only) — reads document
- **Pass 2** : Extract entities with all fields for detected labels — reads document
- **Pass 3** : Translate all entity property values to English — NO document re-reading
- **Pass 4** : Link relationships across ALL entities from ALL chunks — NO document re-reading

**Key design**: Passes 1–3 run per chunk. After all chunks are processed and merged,  
Pass 4 runs **once globally** on the full merged entity set — so cross-document relationships are captured.

**Chunking**: by `=== DOCUMENT X ===`

---

## Bloc 1 — Imports & Configuration

In [1]:
import os
import json
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA

load_dotenv()

print("Imports OK")
print(f"NVIDIA_API_KEY : {'OK' if os.getenv('NVIDIA_API_KEY') else 'MISSING'}")
print(f"NEO4J_URI      : {os.getenv('NEO4J_URI', 'Not defined')}")

Imports OK
NVIDIA_API_KEY : OK
NEO4J_URI      : bolt://localhost:7687


## Bloc 2 — LLM Initialization

In [2]:
llm = ChatNVIDIA(
    model="mistralai/mistral-large-3-675b-instruct-2512",
    api_key=os.getenv("NVIDIA_API_KEY"),
    temperature=0,
    max_completion_tokens=32000,
)

## Bloc 3 — Graph Schema

In [3]:
SCHEMA = """
NODES (entity: fields):
  Company         : name, activity, certifications
  Supplier        : name
  Agreement       : name, description, type, startdate, enddate
  Document        : reference, title, version, date, owner
  Clients         : name, sector
  Clause          : name, description
  governance_body : name, acronyme, role
  Claim           : claim_type, value, unit, scope, source_doc, source_doc_id, quote
  Roles           : title
  Asset           : type, description, classification
  Algorithm       : name, use
  Protocol        : name, use, source
  Risk            : description, level
  Framework       : name, type, use
  Control         : name, requirement, source, Cycle
  Technology      : name, use, source

RELATIONSHIPS (source -[REL]-> target):
  Agreement       -[for]->              Supplier
  Company         -[Use]->              Agreement
  Supplier        -[HAS]->              Company
  Company         -[HAS]->              Document
  Company         -[HAS]->              Clients
  Company         -[HAS]->              governance_body
  Company         -[has_requirement]->  Clause
  Supplier        -[apply_to]->         Clause
  governance_body -[supervise]->        Roles
  governance_body -[approves]->         Claim
  Claim           -[Concerns]->         Asset
  Claim           -[Asserted_by]->      Roles
  Roles           -[operates]->         Technology
  Roles           -[EXECUTES]->         Control
  Algorithm       -[protects]->         Asset
  Protocol        -[used_by]->          Algorithm
  Protocol        -[mitigates]->        Risk
  Risk            -[targeting]->        Asset
  Framework       -[required_by]->      Control
  Control         -[IMPLEMENTED_BY]->   Technology
  Technology      -[Has_access_to]->    Asset
"""

NODE_LABELS = [
    "Company", "Supplier", "Agreement", "Document", "Clients",
    "Clause", "governance_body", "Claim", "Roles", "Asset",
    "Algorithm", "Protocol", "Risk", "Framework", "Control", "Technology"
]

print(f"Schema defined: {len(NODE_LABELS)} node types, 21 relationship types")

Schema defined: 16 node types, 21 relationship types


## Bloc 4 — Prompts for the 4 Passes

**Prompt engineering strategy:**
- **Pass 1** = simple scan → list labels present in the chunk *(reads document)*
- **Pass 2** = targeted extraction → extract detected labels with all fields *(reads document)*
- **Pass 3** = translation → translate all property values to English *(NO document — entities only)*
- **Pass 4** = relationship linking → link entities using schema rules *(NO document — entities only)*

In [4]:
# -----------------------------------------------------------------
# PASS 1 — Which entity types are present in this chunk?
# -----------------------------------------------------------------
PROMPT_PASS1 = f"""
You are a knowledge-graph analyst. Your task is to SCAN a document chunk
and identify which entity types from the schema are mentioned.

Schema node labels:
{', '.join(NODE_LABELS)}

Rules:
- Return ONLY a JSON array of label strings that are present in the text.
- Include a label only if at least one instance is clearly identifiable.
- No explanation, no markdown, no extra text.

Output format example:
["Company", "Document", "Roles", "Control"]
"""

# -----------------------------------------------------------------
# PASS 2 — Extract entities with their fields (targeted labels)
# -----------------------------------------------------------------
def build_prompt_pass2(detected_labels: list) -> str:
    # Filter schema to only show detected types -> shorter, more focused prompt
    schema_lines = []
    for line in SCHEMA.strip().split("\n"):
        for label in detected_labels:
            if line.strip().startswith(label):
                schema_lines.append(line)
                break
    focused_schema = "\n".join(schema_lines)

    # Build the key field reminder per detected label
    key_fields = {
        'Company': 'name', 'Supplier': 'name', 'Agreement': 'name',
        'Document': 'reference', 'Clients': 'name', 'Clause': 'name',
        'governance_body': 'name', 'Claim': 'claim_type', 'Roles': 'title',
        'Asset': 'type', 'Algorithm': 'name', 'Protocol': 'name',
        'Risk': 'description', 'Framework': 'name', 'Control': 'name',
        'Technology': 'name'
    }
    key_reminder = "\n".join(
        f"  - {l}: '{key_fields[l]}' is MANDATORY and must not be empty"
        for l in detected_labels if l in key_fields
    )

    return f"""
You are a knowledge-graph extraction assistant.
Extract ALL entities of the following types from the document chunk.
Be exhaustive: extract every instance you can find.

Target entity types and their fields:
{focused_schema}

MANDATORY KEY FIELDS — each entity MUST have a non-empty value for its key field:
{key_reminder}
If you cannot find a non-empty value for the key field, DO NOT extract that entity.

Rules:
- Use ONLY the node labels listed above.
- Extract every field you can find; omit fields that are not mentioned.
- String values only, no nested objects.
- NEVER invent or hallucinate values — only extract what is explicitly in the text.
- Keep property values in the ORIGINAL language of the document (translation happens in Pass 3).
- Return ONLY a valid JSON array, no markdown, no explanation.

Output format:
[
  {{"label": "<NodeLabel>", "properties": {{"field": "value"}}}},
  ...
]
"""

# -----------------------------------------------------------------
# PASS 3 — Translate all entity property values to English
#           (NO document — operates on extracted entity list only)
# -----------------------------------------------------------------
def build_prompt_pass3_translate(entities: list) -> str:
    entities_json = json.dumps(entities, ensure_ascii=False, indent=2)
    return f"""
You are a translation assistant for a knowledge graph.
Below is a list of extracted entities with their properties.
The property VALUES may be in French or another language.

Your task:
- Translate ALL property VALUES to English — NO EXCEPTIONS.
- This includes job titles, role names, committee names, and governance body names.
  Example: 'Responsable GRC' -> 'GRC Manager', 'Comité Sécurité' -> 'Security Committee',
           'Direction Générale' -> 'General Management', 'Responsable PRA/PCA' -> 'BCP/DRP Manager'
- Keep property KEYS exactly as-is (do not translate field names).
- Keep entity LABELS exactly as-is (do not translate labels).
- Keep proper nouns that are product names or acronyms unchanged (e.g. Azure, SIEM, MFA, ISO 27001).
- If a value is already in English, leave it unchanged.
- Return the SAME JSON structure with translated values only.
- Return ONLY a valid JSON array, no markdown, no explanation.

Entities to translate:
{entities_json}

Output format (same structure, translated values):
[
  {{"label": "<NodeLabel>", "properties": {{"field": "translated value"}}}},
  ...
]
"""

# -----------------------------------------------------------------
# PASS 4 — Link relationships between translated entities
#           (NO document — uses only the translated entity list)
# -----------------------------------------------------------------
def build_prompt_pass4_relations(entities: list) -> str:
    # Build the full entity index with their exact key field value
    key_fields = {
        'Company': 'name', 'Supplier': 'name', 'Agreement': 'name',
        'Document': 'reference', 'Clients': 'name', 'Clause': 'name',
        'governance_body': 'name', 'Claim': 'claim_type', 'Roles': 'title',
        'Asset': 'type', 'Algorithm': 'name', 'Protocol': 'name',
        'Risk': 'description', 'Framework': 'name', 'Control': 'name',
        'Technology': 'name'
    }
    # Build compact entity summary showing ONLY the key field for each entity
    entity_summary = []
    for e in entities:
        label = e['label']
        kf = key_fields.get(label)
        if kf and e['properties'].get(kf):
            entity_summary.append({"label": label, "key": {kf: e['properties'][kf]}})
    entity_summary_json = json.dumps(entity_summary, ensure_ascii=False)
    rel_schema = SCHEMA.split('RELATIONSHIPS')[1].strip()
    return f"""
You are a knowledge-graph relationship extractor.
You are given the EXACT list of entities that exist in the graph.
Your task: identify relationships between them that match the schema.

CRITICAL RULES:
- Do NOT invent or hallucinate entities. Only use entities from the list below.
- from_key and to_key MUST use the EXACT key field and value shown in the list.
  For example, if the list shows {{"label": "governance_body", "key": {{"name": "Security Committee"}}}}
  then from_key or to_key must be {{"name": "Security Committee"}} — never {{"title": "Security Committee"}}.
- Do NOT use fields like 'title' for governance_body (its key is 'name'), or 'description' for Company.
- Do NOT read any source document — work exclusively from the entity list.
- Use ONLY relationship types from the schema.
- Return ONLY a valid JSON array, no markdown, no explanation.

Schema relationships:
{rel_schema}

Entity list (label + exact key field to use in from_key/to_key):
{entity_summary_json}

Output format:
[
  {{
    "from_label": "<NodeLabel>",
    "from_key":   {{"<key_field>": "<exact_value_from_list>"}},
    "rel_type":   "<REL_TYPE>",
    "to_label":   "<NodeLabel>",
    "to_key":     {{"<key_field>": "<exact_value_from_list>"}}
  }},
  ...
]
"""

print("Prompts for all 4 passes defined")

Prompts for all 4 passes defined


## Bloc 5 — Utility Functions

In [5]:
def call_llm(system_prompt: str, user_text: str) -> str:
    """Call the LLM and return raw content."""
    messages = [
        ("system", system_prompt),
        ("human", user_text),
    ]
    raw = llm.invoke(messages).content.strip()
    # Strip markdown fences if present
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()
    return raw


def call_llm_with_doc(system_prompt: str, doc_text: str) -> str:
    """Pass 1 & 2: call LLM with document chunk as user input."""
    return call_llm(system_prompt, f"Document chunk:\n\n{doc_text}")


def call_llm_no_doc(system_prompt: str) -> str:
    """Pass 3 & 4: call LLM without any document.
    The entity data is already embedded in the system_prompt itself."""
    return call_llm(system_prompt, "Process the entities listed in the instructions above.")


def parse_json(raw: str, context: str = ""):
    """Parse JSON with explicit error handling."""
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"      [JSON ERROR] {context} : {e}")
        return None


def split_by_document(text: str, marker: str = "=== DOCUMENT") -> list:
    """
    Split file into chunks by document marker.
    Returns a list of dict {"id": "DOCUMENT X", "text": "..."}
    """
    chunks = []
    parts = text.split(marker)
    for part in parts[1:]:
        first_line = part.split("\n")[0].strip().rstrip("===").strip()
        doc_id = f"DOCUMENT {first_line}"
        chunks.append({"id": doc_id, "text": (marker + part).strip()})
    return chunks if chunks else [{"id": "FULL", "text": text}]


def merge_results(results: list) -> dict:
    """Merge results from all chunks, deduplicated."""
    merged_entities = []
    merged_relations = []
    seen_entities = set()
    seen_relations = set()

    for result in results:
        for entity in result.get("entities", []):
            key = (entity["label"], str(entity.get("properties", {})))
            if key not in seen_entities:
                seen_entities.add(key)
                merged_entities.append(entity)
        for rel in result.get("relationships", []):
            key = (
                rel.get("from_label"),
                str(rel.get("from_key")),
                rel.get("rel_type"),
                rel.get("to_label"),
                str(rel.get("to_key")),
            )
            if key not in seen_relations:
                seen_relations.add(key)
                merged_relations.append(rel)

    return {"entities": merged_entities, "relationships": merged_relations}


print("Utility functions ready")

Utility functions ready


## Bloc 6 — Per-Chunk Pipeline (Pass 1 + 2 + 3 only)

This function runs on each chunk independently and returns translated entities.
**Pass 4 (relationships) is NOT called here** — it runs globally after all chunks are merged.

| Pass | Input | Output |
|------|-------|--------|
| Pass 1 | Document chunk | Detected label list |
| Pass 2 | Document chunk + detected labels | Raw entities (original language) |
| Pass 3 | Raw entities only (no doc) | Translated entities (English) |

In [6]:
def extract_chunk_pass123(chunk: dict, verbose: bool = True) -> list:
    """
    Runs Pass 1, 2, 3 on a single document chunk.
    Returns a list of translated entities (English).
    Pass 4 (relationships) is intentionally excluded — it runs globally
    after ALL chunks are merged so cross-document links are captured.

    Pass 1 : Detect entity types present in the chunk  (reads document)
    Pass 2 : Extract entities with all fields          (reads document)
    Pass 3 : Translate property values to English      (NO document)
    """
    doc_id = chunk["id"]
    text   = chunk["text"]

    if verbose:
        print(f"\n   [{doc_id}] ({len(text):,} chars)")

    # -- PASS 1 : Detect which entity types are present ----------------
    if verbose:
        print(f"      Pass 1 — Label detection...", end=" ", flush=True)

    raw1 = call_llm_with_doc(PROMPT_PASS1, text)
    detected_labels = parse_json(raw1, f"{doc_id} Pass1")

    if not detected_labels or not isinstance(detected_labels, list):
        if verbose:
            print("WARNING: No labels detected, chunk skipped")
        return []

    # Keep only valid schema labels
    detected_labels = [l for l in detected_labels if l in NODE_LABELS]
    if verbose:
        print(f"{len(detected_labels)} labels: {detected_labels}")

    # -- PASS 2 : Extract entities with their fields -------------------
    if verbose:
        print(f"      Pass 2 — Entity extraction...", end=" ", flush=True)

    prompt2 = build_prompt_pass2(detected_labels)
    raw2 = call_llm_with_doc(prompt2, text)
    entities_raw = parse_json(raw2, f"{doc_id} Pass2")

    if not entities_raw or not isinstance(entities_raw, list):
        if verbose:
            print("WARNING: No entities extracted")
        return []

    if verbose:
        print(f"{len(entities_raw)} entities (original language)")

    # -- PASS 3 : Translate entity properties to English ---------------
    #             NO document — works on entity list only
    if verbose:
        print(f"      Pass 3 — Translation to English...", end=" ", flush=True)

    prompt3 = build_prompt_pass3_translate(entities_raw)
    raw3 = call_llm_no_doc(prompt3)
    entities_translated = parse_json(raw3, f"{doc_id} Pass3")

    if not entities_translated or not isinstance(entities_translated, list):
        if verbose:
            print("WARNING: Translation failed, using original entities as fallback")
        return entities_raw  # fallback

    if verbose:
        print(f"{len(entities_translated)} entities translated")

    return entities_translated


print("Per-chunk pipeline (Pass 1+2+3) ready")

Per-chunk pipeline (Pass 1+2+3) ready


## Bloc 7 — Full File Extraction + Global Pass 4

In [7]:
def run_pass4_batched(all_entities: list, batch_size: int = 50) -> list:
    """
    Runs Pass 4 (relationship linking) in batches to avoid LLM timeout.
    With 354 entities a single call exceeds the 302s timeout — batching fixes this.

    Each batch of ~batch_size entities gets its own Pass 4 call.
    Results are merged and deduplicated across all batches.
    """
    all_relationships = []
    seen_relations = set()

    batches = [all_entities[i:i + batch_size] for i in range(0, len(all_entities), batch_size)]
    print(f"   Pass 4 — {len(batches)} batches of ~{batch_size} entities each")

    for i, batch in enumerate(batches, 1):
        print(f"      Batch {i}/{len(batches)} ({len(batch)} entities)...", end=" ", flush=True)
        try:
            prompt4 = build_prompt_pass4_relations(batch)
            raw4 = call_llm_no_doc(prompt4)
            relationships = parse_json(raw4, f"Pass4 batch {i}")

            if not relationships or not isinstance(relationships, list):
                print("no relationships")
                continue

            # Deduplicate across batches
            added = 0
            for rel in relationships:
                key = (
                    rel.get("from_label"),
                    str(rel.get("from_key")),
                    rel.get("rel_type"),
                    rel.get("to_label"),
                    str(rel.get("to_key")),
                )
                if key not in seen_relations:
                    seen_relations.add(key)
                    all_relationships.append(rel)
                    added += 1
            print(f"{added} relationships")

        except Exception as e:
            print(f"ERROR: {e}")

    return all_relationships


def extract_file_4pass(filepath: str, min_chunk_length: int = 200, pass4_batch_size: int = 50) -> dict:
    """
    Loads a file, splits it by document (=== DOCUMENT X ===),
    filters out header-only chunks (table of contents lines < min_chunk_length),
    runs Pass 1+2+3 on each real chunk independently,
    merges all entities (deduplication),
    then runs Pass 4 in batches on the full merged entity set.
    """
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    all_chunks = split_by_document(text)
    # Filter out table-of-contents chunks (title lines only, no real content)
    chunks = [c for c in all_chunks if len(c["text"]) >= min_chunk_length]
    print(f"   {len(all_chunks)} chunks found, {len(chunks)} kept after filtering short ones (< {min_chunk_length} chars)")

    # ── PASS 1 + 2 + 3 : per chunk ────────────────────────────────
    all_entities = []
    seen_entities = set()

    for chunk in chunks:
        try:
            entities = extract_chunk_pass123(chunk, verbose=True)
            for entity in entities:
                key = (entity["label"], str(entity.get("properties", {})))
                if key not in seen_entities:
                    seen_entities.add(key)
                    all_entities.append(entity)
        except Exception as e:
            print(f"      WARNING: Error on {chunk['id']} : {e}")

    print(f"\n   Pass 1-3 complete: {len(all_entities)} unique entities (translated, English)")

    # ── PASS 4 : batched relationship linking ─────────────────────
    relationships = run_pass4_batched(all_entities, batch_size=pass4_batch_size)

    print(f"   DONE: {len(all_entities)} entities, {len(relationships)} relationships")
    return {"entities": all_entities, "relationships": relationships}


print("Full extraction function ready (with batched Pass 4 + short-chunk filter)")

Full extraction function ready (with batched Pass 4 + short-chunk filter)


## Bloc 8 — Run Extraction

In [8]:
DOCUMENTS = [
    "/home/maroua/Desktop/cassiope/SecuraOps High.txt",
    # "/home/maroua/Desktop/cassiope/SafeLink Contradictory.txt",
]

raw_results = {}

for filepath in DOCUMENTS:
    nom = filepath.split("/")[-1]
    print(f"\n{'='*60}")
    print(f"Extraction: {nom}")
    print(f"{'='*60}")
    try:
        result = extract_file_4pass(filepath)
        raw_results[filepath] = result
    except Exception as e:
        print(f"ERROR: {e}")
        raw_results[filepath] = {"entities": [], "relationships": []}


Extraction: SecuraOps High.txt
   44 chunks found, 22 kept after filtering short ones (< 200 chars)

   [DOCUMENT 1 : Politique de Sécurité de l’Information] (9,640 chars)
      Pass 1 — Label detection... 10 labels: ['Company', 'Document', 'Clients', 'Clause', 'governance_body', 'Asset', 'Risk', 'Framework', 'Control', 'Technology']
      Pass 2 — Entity extraction...       WARNING: Error on DOCUMENT 1 : Politique de Sécurité de l’Information : [504] Gateway Timeout
{'_content': b'', '_content_consumed': True, '_next': None, 'status_code': 504, 'headers': {'Date': 'Sun, 05 Apr 2026 18:10:07 GMT', 'Content-Length': '0', 'Connection': 'keep-alive', 'Access-Control-Expose-Headers': 'nvcf-reqid', 'Nvcf-Reqid': 'fae9289a-b871-4aca-9df2-aaafb4b2177f', 'Nvcf-Status': 'errored', 'Vary': 'Origin'}, 'raw': <urllib3.response.HTTPResponse object at 0x77a44a7d2b00>, 'url': 'https://integrate.api.nvidia.com/v1/chat/completions', 'encoding': None, 'history': [], 'reason': 'Gateway Timeout', 'cookie

## Bloc 9 — Results Visualization

In [9]:
from collections import Counter

for filepath, result in raw_results.items():
    nom = filepath.split("/")[-1]
    print(f"\n{nom}")
    print(f"   Entities      : {len(result['entities'])}")
    print(f"   Relationships : {len(result['relationships'])}")
    node_counts = Counter(e['label'] for e in result['entities'])
    print("   Entity types:")
    for label, count in sorted(node_counts.items()):
        print(f"      {label:<20} : {count}")

    # Sample check: show first 5 entities to verify translation quality
    print("\n   Sample entities (first 5, should be in English):")
    for e in result['entities'][:5]:
        print(f"      [{e['label']}] {e['properties']}")


SecuraOps High.txt
   Entities      : 324
   Relationships : 83
   Entity types:
      Agreement            : 3
      Asset                : 11
      Clients              : 2
      Company              : 14
      Control              : 108
      Cycle                : 1
      Document             : 14
      Framework            : 18
      Protocol             : 10
      Risk                 : 14
      Roles                : 39
      Technology           : 86
      governance_body      : 4

   Sample entities (first 5, should be in English):
      [Company] {'name': 'SecuraOps'}
      [Technology] {'name': 'Azure Key Vault', 'use': 'API secret and technical account management'}
      [Technology] {'name': 'PowerShell', 'use': 'Automation of secret rotations', 'source': 'scripts'}
      [Protocol] {'name': 'SHA-512', 'use': 'Password storage in salted hash form', 'source': 'salted hash'}
      [Protocol] {'name': 'MFA', 'use': 'Multi-factor authentication for all privileged access'}


## Bloc 10 — JSON Save + Auto Post-Processing

In [10]:
from copy import deepcopy

# ── Post-processing helpers ────────────────────────────────────
KEY_FIELDS = {
    'Company': 'name', 'Supplier': 'name', 'Agreement': 'name',
    'Document': 'reference', 'Clients': 'name', 'Clause': 'name',
    'governance_body': 'name', 'Claim': 'claim_type', 'Roles': 'title',
    'Asset': 'type', 'Algorithm': 'name', 'Protocol': 'name',
    'Risk': 'description', 'Framework': 'name', 'Control': 'name',
    'Technology': 'name'
}

def merge_props(a, b):
    m = deepcopy(a)
    for k, v in b.items():
        if v and (not m.get(k) or len(str(v)) > len(str(m.get(k,'')))):
            m[k] = v
    return m

def postprocess(result: dict) -> dict:
    entities = result['entities']
    relationships = result['relationships']
    print(f"  Before: {len(entities)} entities, {len(relationships)} relationships")

    # 1. Deduplicate entities
    seen, deduped = {}, []
    for e in entities:
        kf = KEY_FIELDS.get(e['label'])
        kv = e['properties'].get(kf, '').strip().lower() if kf else ''
        dk = (e['label'], kv) if kv else None
        if dk and dk in seen:
            deduped[seen[dk]]['properties'] = merge_props(deduped[seen[dk]]['properties'], e['properties'])
        else:
            deduped.append(deepcopy(e))
            if dk: seen[dk] = len(deduped) - 1
    print(f"  Dedup : {len(entities)} → {len(deduped)} entities")

    # 2. Drop entities with empty key fields
    valid_entities = []
    for e in deduped:
        kf = KEY_FIELDS.get(e['label'])
        if kf and not e['properties'].get(kf, '').strip():
            print(f"  Drop  : [{e['label']}] {e['properties']}")
        else:
            valid_entities.append(e)

    # 3. Build index and remove broken relationships
    index = set()
    for e in valid_entities:
        index.add((e['label'], frozenset(e['properties'].items())))

    def matches(label, key_dict):
        return any(l == label and all(dict(p).get(k) == v for k, v in key_dict.items())
                   for l, p in index)

    valid_rels = [r for r in relationships
                  if matches(r['from_label'], r['from_key']) and matches(r['to_label'], r['to_key'])]
    print(f"  Rels  : {len(relationships)} → {len(valid_rels)} (removed {len(relationships)-len(valid_rels)} broken)")

    print(f"  After : {len(valid_entities)} entities, {len(valid_rels)} relationships")
    return {'entities': valid_entities, 'relationships': valid_rels}


# ── Save raw + clean JSON for each file ───────────────────────
for filepath, result in raw_results.items():
    nom = filepath.split('/')[-1].replace('.txt', '')

    # Raw output
    raw_file = f"{nom}_graph_4pass.json"
    with open(raw_file, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    print(f"Raw saved   : {raw_file}")

    # Clean output
    print(f"Post-processing {nom}...")
    clean = postprocess(result)
    clean_file = f"{nom}_graph_4pass_clean.json"
    with open(clean_file, 'w', encoding='utf-8') as f:
        json.dump(clean, f, indent=2, ensure_ascii=False)
    print(f"Clean saved : {clean_file}\n")

Raw saved   : SecuraOps High_graph_4pass.json
Post-processing SecuraOps High...
  Before: 324 entities, 83 relationships
  Dedup : 324 → 304 entities
  Drop  : [Document] {'name': 'Vulnerability Management Policy', 'source': 'VULN-PO-001'}
  Drop  : [Clients] {'name': ''}
  Drop  : [Document] {'name': 'AWARE-PO-001'}
  Rels  : 83 → 77 (removed 6 broken)
  After : 301 entities, 77 relationships
Clean saved : SecuraOps High_graph_4pass_clean.json



## Bloc 11 — Cypher Query Generation

In [ ]:
def props_str(props: dict) -> str:
    parts = [f"{k}: {json.dumps(v)}" for k, v in props.items()]
    return "{" + ", ".join(parts) + "}"


def json_to_cypher(extracted: dict, company_label: str = None) -> list:
    statements = []
    for entity in extracted.get("entities", []):
        label = entity["label"]
        props = entity.get("properties", {})
        if props:
            if company_label:
                statements.append(f"MERGE (n:{label}:{company_label} {props_str(props)});")
            else:
                statements.append(f"MERGE (n:{label} {props_str(props)});")
    for rel in extracted.get("relationships", []):
        fl = rel["from_label"]
        fk = props_str(rel["from_key"])
        rt = rel["rel_type"]
        tl = rel["to_label"]
        tk = props_str(rel["to_key"])
        if company_label:
            statements.append(
                f"MATCH (a:{fl}:{company_label} {fk}), (b:{tl}:{company_label} {tk}) MERGE (a)-[:{rt}]->(b);"
            )
        else:
            statements.append(
                f"MATCH (a:{fl} {fk}), (b:{tl} {tk}) MERGE (a)-[:{rt}]->(b);"
            )
    return statements


company_map = {
    "SecuraOps High": "SecuraOps",
    "SafeLink Contradictory": "SafeLink",
}

cypher_by_file = {}
for filepath, result in raw_results.items():
    nom = filepath.split("/")[-1].replace(".txt", "")
    company = company_map.get(nom, nom)
    stmts = json_to_cypher(result, company_label=company)
    cypher_by_file[nom] = stmts
    print(f"OK {nom} : {len(stmts)} Cypher statements")

## Bloc 12 — Send to Neo4j

In [ ]:
from neo4j import GraphDatabase

uri      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
user     = os.getenv("NEO4J_USER",     "neo4j")
password = os.getenv("NEO4J_PASSWORD", "")

driver = GraphDatabase.driver(uri, auth=(user, password))

# Clear the database
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
print("Database cleared")

# Send statements
for nom, stmts in cypher_by_file.items():
    print(f"\nSending {nom} to Neo4j...")
    errors = 0
    with driver.session() as session:
        for stmt in stmts:
            try:
                session.run(stmt)
            except Exception as e:
                errors += 1
    print(f"   {len(stmts) - errors} statements OK")
    if errors:
        print(f"   {errors} errors")

driver.close()
print("\nDone!")